In [1]:
import glob
import ast
import time
import numpy as np
import pandas as pd
from tqdm import tqdm 
import cv2
import json
import collections
from PIL import Image
import re
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import zipfile

from tqdm import tqdm
import shutil

np.random.seed(42)

In [2]:
import warnings
warnings.filterwarnings('ignore')

### Verify GPU

In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch_type = torch.float32 if device.type == "cuda" else torch.float16
device, torch_type

(device(type='cuda'), torch.float32)

In [4]:
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA GeForce RTX 3090
Memory: 25.43 GB


### Loading package

In [5]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[1]
sys.path.append(str(repo_path))

In [6]:
from py.utils import verifyDir,verifyFile, verifyDataFrame

In [7]:
from py.config import Config

cfg = Config()

np.random.seed(cfg.RANDOM_STATE)
cfg.DATA_PATH, cfg.MODEL_PATH

('/media/felipe/DATA19/datasets/', '/media/felipe/DATA19/models/')

In [8]:
OUT_NAME = f"{cfg.PERCEPTION_METRIC}"
OUT_NAME += "_filter" if cfg.FILTER_FEATURES else ""
OUT_NAME += "_bin" if cfg.BINARIZE_FEATURES else ""
OUT_NAME

'safety'

In [9]:
CF_DIR = f"{cfg.MODEL_PATH}"
CF_DIR += f"{cfg.SEG_DATASET}_upd4k" if cfg.USE_UPD else f"{cfg.SEG_DATASET}"
CF_DIR += "_group/" if cfg.BY_GROUPS else "/"
CF_DIR += "" if cfg.USE_UPD else f"{cfg.SEG_MODEL_NAME}/"
CF_DIR += f"counterfactuals/{OUT_NAME}/"
CF_DIR

'/media/felipe/DATA19/models/ade20k_upd4k_group/counterfactuals/safety/'

### Loading Data

In [10]:
cf_test_data_df = pd.read_csv(f"{CF_DIR}cf_test_data.csv", sep=";", low_memory=False)

In [11]:
features_name = cf_test_data_df.iloc[:, 1:-2].columns.tolist()

# LLM Interpretations

In [12]:
from py.counterfactuals import CounterfactualAnalyzer

In [13]:
cf_analyzer = CounterfactualAnalyzer()
cf_analyzer.load(CF_DIR)

In [14]:
results = cf_analyzer.get_results()
unsafe2safe_df = results["nearest_cf_variation"].copy()
unsafe2safe_df["diff_new_prob"] = unsafe2safe_df["diff_prob"].apply(lambda x: x[0][1])
unsafe2safe_df.sort_values(by="diff_new_prob", ascending=False, inplace=True)
unsafe2safe_df.head(30)

,city_elements,construction,floor,human,sky,terrain_vehicle,vegetation,broken_damaged_bricks_wall,broken_damaged_pavement_road,broken_window,...,trashcan,image_id,orig_prob,new_prob,diff_prob,orig_class,new_class,desired_class_prob,euclidean_dist,diff_new_prob
463,0.000000,0.000000,0.000000,0.000000,-29.092500,0.00,0.000000,-3.890000,0.000000,0.0,...,0.0,50f5eaaafdc9f065f0007bd5,"[[0.89944444, 0.10055556]]","[[0.26603571, 0.73396429]]","[[-0.63340873, 0.63340873]]",0,1,0.733964,29.355675,0.633409
335,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,-6.319167,-4.537500,0.0,...,0.0,50f5ec13fdc9f065f00086c6,"[[0.89158856, 0.10841144]]","[[0.26214881, 0.73785119]]","[[-0.62943975, 0.62943975]]",0,1,0.737851,7.802613,0.629440
581,0.000000,0.000000,0.000000,0.000000,-17.353333,-0.15,0.000000,0.000000,0.000000,0.0,...,0.0,50f5ebaafdc9f065f000843a,"[[0.6938141, 0.3061859]]","[[0.11915675, 0.88084325]]","[[-0.57465736, 0.57465736]]",0,1,0.880843,17.353982,0.574657
650,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,-9.331667,0.000000,0.0,...,0.0,50f5ebcefdc9f065f0008595,"[[0.90021429, 0.09978571]]","[[0.32810516, 0.67189484]]","[[-0.57210913, 0.57210913]]",0,1,0.671895,9.350936,0.572109
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,-8.041667,0.000000,0.0,...,0.0,50f5ebc8fdc9f065f000851e,"[[0.90663095, 0.09336905]]","[[0.33884325, 0.66115675]]","[[-0.5677877, 0.5677877]]",0,1,0.661157,8.064019,0.567788
543,0.000000,0.000000,0.000000,0.000000,-18.143333,0.00,0.000000,0.000000,0.000000,0.0,...,0.0,50f5ec18fdc9f065f000871d,"[[0.75085324, 0.24914676]]","[[0.18406746, 0.81593254]]","[[-0.56678578, 0.56678578]]",0,1,0.815933,18.153252,0.566786
333,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,-3.847500,0.000000,0.0,...,0.0,50f5ec3ffdc9f065f00088a9,"[[0.88987103, 0.11012897]]","[[0.33755556, 0.66244444]]","[[-0.55231548, 0.55231548]]",0,1,0.662444,3.894003,0.552315
278,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,-22.030833,0.000000,0.0,...,0.0,50f5ec16fdc9f065f00086f8,"[[0.81600595, 0.18399405]]","[[0.28458135, 0.71541865]]","[[-0.5314246, 0.5314246]]",0,1,0.715419,22.039002,0.531425
260,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,-13.140833,0.000000,0.0,...,0.0,50f5eba3fdc9f065f00083b4,"[[0.77852976, 0.22147024]]","[[0.25332998, 0.74667002]]","[[-0.52519979, 0.52519979]]",0,1,0.746670,13.154524,0.525200
237,0.000000,0.000000,0.000000,0.600000,0.000000,0.60,0.000000,-3.427500,0.000000,0.0,...,0.6,50f5eaacfdc9f065f0007bf7,"[[0.94769246, 0.05230754]]","[[0.42471771, 0.57528229]]","[[-0.52297475, 0.52297475]]",0,1,0.575282,5.676833,0.522975


### Prompt

In [15]:
from py.LLM import Chat, QuantizedChat

In [16]:
# llm_chat = QuantizedChat(model_name="Qwen/Qwen2.5-3B-Instruct")
llm_chat = Chat(model_name="Qwen/Qwen2.5-3B-Instruct")

Loading model: Qwen/Qwen2.5-3B-Instruct...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded successfully on cuda


In [17]:
llm_chat.set_system_message(
    "You are an expert in urban planning and visual perception.\n" \
    "Your task is to explain how changes in specific visual elements of a street scene might influence the way people perceive safety."
)

In [18]:
%%time
llm_interpretations = []

for index, row_ in tqdm(unsafe2safe_df.iterrows()):
    changes = row_[features_name].to_frame().T
    # increase
    #added = changes.gt(0).apply(lambda r: r.index[r].tolist(), axis=1)
    added = changes.columns[(changes > 0).any() & (changes > cfg.CHANGE_THRESHOLD).any()].tolist()
    increased = [ f"{a_}: increased in { round( changes[[a_]].values[0][0] , 3) }" for a_ in added ]
    # decrease
    #substracted = changes.lt(0).apply(lambda r: r.index[r].tolist(), axis=1)
    substracted = changes.columns[(changes < 0).any() & (changes < -1*cfg.CHANGE_THRESHOLD).any()].tolist()
    decreased = [ f"{a_}: decreased in { round( abs(changes[[a_]].values[0][0]) , 3) }" for a_ in substracted ]

    prompt_changes = f"""
    We have a list of visual elements added or substracted from an image.
    These changes come from comparing the original image with a generated counterfactual version.
    [Added elements]:
    -{chr(10).join(increased) if increased else "None"}
    [Removed elements]:
    -{chr(10).join(decreased) if decreased else "None"}
    Please write a concise explanation (2–4 sentences) describing the elements to add and remove and how these changes may affect safety perception.
    """

    # Get response from the model
    response = llm_chat.chat(prompt_changes, temperature=0.7, max_new_tokens=256)
    
    # Store the result
    llm_interpretations.append({
        'index': index,
        'increased': increased,
        'decreased': decreased,
        'explanation': response
    })
    
    print(f"\n--- Row {index} ---")
    print(f"Response: {response}\n")
    break

0it [00:01, ?it/s]


--- Row 463 ---
Response: The addition of overhead cables by increasing their presence can potentially enhance perceived safety, as they provide clear visual cues about electrical hazards and can deter unauthorized activities. Conversely, removing sky elements, such as overcast skies or distant clouds, which can often symbolize openness and tranquility, could reduce this calming effect and potentially lead to a perception of reduced safety and more confined spaces. Removing broken and damaged bricks from walls could also improve safety perception by suggesting a maintained and secure environment, reducing concerns about potential structural issues.

CPU times: user 1.44 s, sys: 195 ms, total: 1.63 s
Wall time: 1.63 s
